# 🔧 Data Preprocessing Pipeline

## Supply Chain Late Delivery Prediction

---

### Overview

This notebook transforms raw supply chain data into a clean, ML-ready format.

### Preprocessing Steps

| Step | Action | Method |
|------|--------|--------|
| 1 | Standardize columns | Lowercase, underscores, remove special chars |
| 2 | Parse dates | Convert to datetime objects |
| 3 | Handle missing values | Drop high-null columns, impute others |
| 4 | Remove duplicates | Drop exact duplicate rows |
| 5 | Cap outliers | IQR-based capping for numeric features |
| 6 | Verify target | Ensure binary late_delivery variable |

---

### Data Quality Rules

1. **Drop columns with >90% missing**: `product_description`
2. **Impute numeric**: Median (robust to outliers)
3. **Impute categorical**: Mode (most frequent value)
4. **Outlier treatment**: Cap at 1.5x IQR bounds

---

In [15]:
# ============================================================
# SETUP & DATA LOADING
# ============================================================
import sys
import warnings
warnings.filterwarnings('ignore')
sys.path.append('..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from src.data.data_manager import load_raw

import plotly.io as pio
pio.templates.default = "plotly_white"

print("Libraries loaded successfully")

Libraries loaded successfully


In [16]:
# Load raw data
df_raw = load_raw()

# Store original shape for comparison
original_shape = df_raw.shape
original_missing = df_raw.isnull().sum().sum()
original_memory = df_raw.memory_usage(deep=True).sum() / 1024**2

print(f"Raw Data Loaded:")
print(f"   Rows: {original_shape[0]:,}")
print(f"   Columns: {original_shape[1]}")
print(f"   Missing values: {original_missing:,}")
print(f"   Memory: {original_memory:.1f} MB")

📂 Loading raw file: /Users/unclesam/Projects/supply-chain-ml-project/data/raw/DataCoSupplyChainDataset.csv
⚠️ UTF-8 decode failed. Retrying with Latin-1...
Raw Data Loaded:
   Rows: 180,519
   Columns: 53
   Missing values: 336,209
   Memory: 332.6 MB


---

## 1. Column Standardization

**Talking Point**: "First, we standardize column names for consistent access throughout the pipeline."

In [17]:
# ============================================================
# STEP 1: COLUMN STANDARDIZATION
# ============================================================

# Create working copy
df = df_raw.copy()

# Standardize column names
original_cols = df.columns.tolist()
df.columns = (df.columns
              .str.strip()
              .str.lower()
              .str.replace(' ', '_')
              .str.replace('(', '')
              .str.replace(')', '')
              .str.replace('-', '_'))

new_cols = df.columns.tolist()

# Show changes
changes = [(orig, new) for orig, new in zip(original_cols, new_cols) if orig != new]
print(f"Column names standardized: {len(changes)} columns renamed")
print(f"\nSample changes (first 10):")
for orig, new in changes[:10]:
    print(f"   '{orig}' -> '{new}'")

Column names standardized: 53 columns renamed

Sample changes (first 10):
   'Type' -> 'type'
   'Days for shipping (real)' -> 'days_for_shipping_real'
   'Days for shipment (scheduled)' -> 'days_for_shipment_scheduled'
   'Benefit per order' -> 'benefit_per_order'
   'Sales per customer' -> 'sales_per_customer'
   'Delivery Status' -> 'delivery_status'
   'Late_delivery_risk' -> 'late_delivery_risk'
   'Category Id' -> 'category_id'
   'Category Name' -> 'category_name'
   'Customer City' -> 'customer_city'


---

## 2. Date Parsing

**Talking Point**: "Date columns need proper datetime parsing for temporal feature extraction."

In [18]:
# ============================================================
# STEP 2: DATE PARSING
# ============================================================

# Identify potential date columns
date_cols = [c for c in df.columns if 'date' in c.lower()]
print(f"Date columns identified: {date_cols}")

# Parse date columns
for col in date_cols:
    if df[col].dtype == 'object':
        try:
            df[col] = pd.to_datetime(df[col], errors='coerce')
            valid_dates = df[col].notna().sum()
            print(f"   {col}: Parsed {valid_dates:,} valid dates")
        except Exception as e:
            print(f"   {col}: Could not parse - {e}")
    else:
        print(f"   {col}: Already datetime type")

Date columns identified: ['order_date_dateorders', 'shipping_date_dateorders']
   order_date_dateorders: Parsed 180,519 valid dates
   shipping_date_dateorders: Parsed 180,519 valid dates


---

## 3. Missing Value Analysis & Treatment

**Talking Point**: "Missing data requires careful handling. We'll drop columns with excessive nulls and impute the rest strategically."

In [19]:
# ============================================================
# STEP 3: MISSING VALUE ANALYSIS
# ============================================================

# Calculate missing value statistics
missing_stats = pd.DataFrame({
    'column': df.columns,
    'missing_count': df.isnull().sum().values,
    'missing_pct': (df.isnull().sum() / len(df) * 100).values,
    'dtype': df.dtypes.values
}).sort_values('missing_pct', ascending=False)

# Filter to columns with missing values
missing_cols = missing_stats[missing_stats['missing_count'] > 0]

if len(missing_cols) > 0:
    # Visualize missing values
    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=missing_cols['missing_pct'].values,
        y=missing_cols['column'].values,
        orientation='h',
        marker_color=np.where(missing_cols['missing_pct'] > 50, '#e74c3c',
                              np.where(missing_cols['missing_pct'] > 10, '#f39c12', '#2ecc71')),
        text=[f"{v:.1f}%" for v in missing_cols['missing_pct']],
        textposition='outside'
    ))

    fig.update_layout(
        title='<b>Missing Values by Column</b>',
        xaxis_title='Missing Percentage',
        yaxis={'categoryorder': 'total ascending'},
        height=max(400, len(missing_cols) * 25),
        showlegend=False
    )

    # Add threshold lines
    fig.add_vline(x=50, line_dash="dash", line_color="red",
                  annotation_text="Drop threshold (50%)")
    fig.add_vline(x=10, line_dash="dash", line_color="orange",
                  annotation_text="High missing (10%)")
    fig.show()

    # Dynamic interpretation
    high_missing = missing_cols[missing_cols['missing_pct'] > 50]['column'].tolist()
    moderate_missing = missing_cols[(missing_cols['missing_pct'] > 10) &
                                    (missing_cols['missing_pct'] <= 50)]['column'].tolist()
    low_missing = missing_cols[missing_cols['missing_pct'] <= 10]['column'].tolist()

    print(f"\nMISSING VALUE ANALYSIS:")
    print(f"   Total columns with missing: {len(missing_cols)}")
    print(f"   High missing (>50%, will drop): {len(high_missing)} - {high_missing}")
    print(f"   Moderate missing (10-50%): {len(moderate_missing)} - {moderate_missing}")
    print(f"   Low missing (<10%): {len(low_missing)}")
else:
    print("No missing values found in the dataset!")
    high_missing = []


MISSING VALUE ANALYSIS:
   Total columns with missing: 4
   High missing (>50%, will drop): 2 - ['product_description', 'order_zipcode']
   Moderate missing (10-50%): 0 - []
   Low missing (<10%): 2


In [20]:
# ============================================================
# STEP 3B: MISSING VALUE TREATMENT
# ============================================================

# Drop columns with >50% missing
cols_to_drop = missing_stats[missing_stats['missing_pct'] > 50]['column'].tolist()
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped {len(cols_to_drop)} high-null columns: {cols_to_drop}")

# Impute remaining missing values
imputation_log = []

for col in df.columns:
    missing_count = df[col].isnull().sum()
    if missing_count > 0:
        dtype = df[col].dtype

        if pd.api.types.is_numeric_dtype(dtype):
            # Numeric: impute with median
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            imputation_log.append({'column': col, 'method': 'median', 'value': median_val, 'count': missing_count})
        else:
            # Categorical: impute with mode
            mode_val = df[col].mode().iloc[0] if len(df[col].mode()) > 0 else 'Unknown'
            df[col] = df[col].fillna(mode_val)
            imputation_log.append({'column': col, 'method': 'mode', 'value': mode_val, 'count': missing_count})

if imputation_log:
    imputation_df = pd.DataFrame(imputation_log)
    print(f"\nImputed {len(imputation_log)} columns:")
    print(imputation_df.to_string(index=False))

# Verify no missing values remain
remaining_missing = df.isnull().sum().sum()
print(f"\nRemaining missing values: {remaining_missing}")

Dropped 2 high-null columns: ['product_description', 'order_zipcode']

Imputed 2 columns:
          column method    value  count
  customer_lname   mode    Smith      8
customer_zipcode median  19380.0      3

Remaining missing values: 0


---

## 4. Duplicate Detection & Removal

**Talking Point**: "Duplicate records can bias our model. We check for and remove exact duplicates."

In [21]:
# ============================================================
# STEP 4: DUPLICATE REMOVAL
# ============================================================

# Check for duplicates
n_duplicates = df.duplicated().sum()
dup_pct = n_duplicates / len(df) * 100

print(f"Duplicate Analysis:")
print(f"   Duplicate rows found: {n_duplicates:,} ({dup_pct:.2f}%)")

if n_duplicates > 0:
    df = df.drop_duplicates()
    print(f"   Duplicates removed. New row count: {len(df):,}")
else:
    print(f"   No duplicates found")

Duplicate Analysis:
   Duplicate rows found: 0 (0.00%)
   No duplicates found


---

## 5. Outlier Detection & Treatment

**Talking Point**: "Outliers can distort model training. We use IQR-based capping to handle extreme values while preserving the underlying distribution."

In [22]:
# ============================================================
# STEP 5: OUTLIER DETECTION & TREATMENT
# ============================================================

# Select numeric columns for outlier analysis
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Exclude ID columns and specific columns
exclude_patterns = ['_id', 'id_', 'latitude', 'longitude', 'zipcode', 'late_delivery']
numeric_cols = [c for c in numeric_cols if not any(p in c.lower() for p in exclude_patterns)]

# Calculate outlier statistics
outlier_stats = []
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    outlier_pct = outliers / len(df) * 100

    if outliers > 0:
        outlier_stats.append({
            'column': col,
            'outliers': outliers,
            'outlier_pct': outlier_pct,
            'lower_bound': lower_bound,
            'upper_bound': upper_bound
        })

outlier_df = pd.DataFrame(outlier_stats).sort_values('outlier_pct', ascending=False)

if len(outlier_df) > 0:
    # Visualize outlier distribution
    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=outlier_df['outlier_pct'].values[:15],
        y=outlier_df['column'].values[:15],
        orientation='h',
        marker_color='#e74c3c',
        text=[f"{v:.1f}%" for v in outlier_df['outlier_pct'].values[:15]],
        textposition='outside'
    ))

    fig.update_layout(
        title='<b>Outlier Percentage by Column (Top 15)</b>',
        xaxis_title='Outlier Percentage',
        yaxis={'categoryorder': 'total ascending'},
        height=500,
        showlegend=False
    )
    fig.show()

    print(f"\nOUTLIER ANALYSIS:")
    print(f"   Columns with outliers: {len(outlier_df)}")
    print(f"   Max outlier percentage: {outlier_df['outlier_pct'].max():.1f}%")


OUTLIER ANALYSIS:
   Columns with outliers: 9
   Max outlier percentage: 10.5%


In [23]:
# Cap outliers using IQR method
capping_log = []

for _, row in outlier_df.iterrows():
    col = row['column']
    lower = row['lower_bound']
    upper = row['upper_bound']

    # Only cap if there are significant outliers (>0.5%)
    if row['outlier_pct'] > 0.5:
        before_min, before_max = df[col].min(), df[col].max()
        df[col] = df[col].clip(lower=lower, upper=upper)
        after_min, after_max = df[col].min(), df[col].max()

        capping_log.append({
            'column': col,
            'before_range': f"{before_min:.2f} to {before_max:.2f}",
            'after_range': f"{after_min:.2f} to {after_max:.2f}",
            'outliers_capped': int(row['outliers'])
        })

if capping_log:
    capping_df = pd.DataFrame(capping_log)
    print(f"\nOutliers capped in {len(capping_log)} columns:")
    print(capping_df.to_string(index=False))


Outliers capped in 8 columns:
                  column       before_range      after_range  outliers_capped
       benefit_per_order -4274.98 to 911.80 -79.70 to 151.50            18942
  order_profit_per_order -4274.98 to 911.80 -79.70 to 151.50            18942
 order_item_profit_ratio      -2.75 to 0.50    -0.34 to 0.50            17300
     order_item_discount     0.00 to 500.00    0.00 to 66.87             7537
order_item_product_price    9.99 to 1999.99   9.99 to 424.98             2048
           product_price    9.99 to 1999.99   9.99 to 424.98             2048
      sales_per_customer    7.49 to 1939.99   7.49 to 461.93             1943
        order_item_total    7.49 to 1939.99   7.49 to 461.93             1943


---

## 6. Target Variable Verification

**Talking Point**: "Let's verify our target variable is properly formatted as binary (0/1) for classification."

In [24]:
# ============================================================
# STEP 6: TARGET VARIABLE VERIFICATION
# ============================================================

# Find target column
target_col = None
for col in ['late_delivery_risk', 'late_delivery', 'latedeliveryrisk']:
    if col in df.columns:
        target_col = col
        break

if target_col:
    # Ensure binary format
    df[target_col] = df[target_col].astype(int)

    target_dist = df[target_col].value_counts()
    target_pct = df[target_col].value_counts(normalize=True) * 100

    print(f"TARGET VARIABLE: {target_col}")
    print(f"   Values: {target_dist.to_dict()}")
    print(f"   Late deliveries: {target_dist.get(1, 0):,} ({target_pct.get(1, 0):.1f}%)")
    print(f"   On-time deliveries: {target_dist.get(0, 0):,} ({target_pct.get(0, 0):.1f}%)")

    # Verify binary
    unique_vals = df[target_col].unique()
    if set(unique_vals) == {0, 1}:
        print(f"   Target is properly binary (0, 1)")
    else:
        print(f"   Warning: Unexpected values: {unique_vals}")
else:
    print("Warning: Target variable not found!")

TARGET VARIABLE: late_delivery_risk
   Values: {1: 98977, 0: 81542}
   Late deliveries: 98,977 (54.8%)
   On-time deliveries: 81,542 (45.2%)
   Target is properly binary (0, 1)


---

## 7. Preprocessing Summary & Comparison

**Talking Point**: "Here's a before/after comparison showing the impact of our preprocessing pipeline."

In [25]:
# ============================================================
# PREPROCESSING SUMMARY
# ============================================================

# Calculate final statistics
final_shape = df.shape
final_missing = df.isnull().sum().sum()
final_memory = df.memory_usage(deep=True).sum() / 1024**2

# Create comparison
comparison_data = {
    'Metric': ['Rows', 'Columns', 'Missing Values', 'Memory (MB)'],
    'Before': [f"{original_shape[0]:,}", original_shape[1], f"{original_missing:,}", f"{original_memory:.1f}"],
    'After': [f"{final_shape[0]:,}", final_shape[1], f"{final_missing:,}", f"{final_memory:.1f}"],
    'Change': [
        f"{final_shape[0] - original_shape[0]:+,}",
        f"{final_shape[1] - original_shape[1]:+}",
        f"{final_missing - original_missing:+,}",
        f"{final_memory - original_memory:+.1f}"
    ]
}
comparison_df = pd.DataFrame(comparison_data)

print("\n" + "=" * 60)
print("PREPROCESSING COMPARISON: BEFORE vs AFTER")
print("=" * 60)
print(comparison_df.to_string(index=False))


PREPROCESSING COMPARISON: BEFORE vs AFTER
        Metric  Before   After   Change
          Rows 180,519 180,519       +0
       Columns      53      51       -2
Missing Values 336,209       0 -336,209
   Memory (MB)   332.6   309.0    -23.6


In [26]:
# Visual comparison
fig = make_subplots(
    rows=1, cols=3,
    specs=[[{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}]],
    subplot_titles=('Rows Retained', 'Missing Values Resolved', 'Columns')
)

# Rows retained
retention_pct = final_shape[0] / original_shape[0] * 100
fig.add_trace(go.Indicator(
    mode="gauge+number",
    value=retention_pct,
    number={'suffix': '%'},
    gauge={
        'axis': {'range': [0, 100]},
        'bar': {'color': '#2ecc71'},
        'threshold': {'line': {'color': 'red', 'width': 4}, 'value': 95}
    }
), row=1, col=1)

# Missing values reduction
missing_reduction = (1 - final_missing / original_missing) * 100 if original_missing > 0 else 100
fig.add_trace(go.Indicator(
    mode="gauge+number",
    value=missing_reduction,
    number={'suffix': '%'},
    gauge={
        'axis': {'range': [0, 100]},
        'bar': {'color': '#3498db'},
    }
), row=1, col=2)

# Columns
fig.add_trace(go.Indicator(
    mode="number+delta",
    value=final_shape[1],
    delta={'reference': original_shape[1], 'relative': False},
    number={'font': {'size': 40}}
), row=1, col=3)

fig.update_layout(
    height=300,
    title='<b>Preprocessing Impact Dashboard</b>'
)
fig.show()

In [27]:
# Save preprocessed data
from pathlib import Path
from datetime import datetime

output_dir = Path('../data/interim')
output_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
output_path = output_dir / f'cleaned_data_{timestamp}.parquet'

df.to_parquet(output_path, index=False)
print(f"\nPreprocessed data saved to: {output_path}")
print(f"   File size: {output_path.stat().st_size / 1024**2:.1f} MB")


Preprocessed data saved to: ../data/interim/cleaned_data_20251205_1120.parquet
   File size: 9.5 MB


In [28]:
# Final summary
print("\n" + "=" * 80)
print("DATA PREPROCESSING COMPLETE")
print("=" * 80)

print(f"""
PREPROCESSING PIPELINE SUMMARY
{'='*60}

1. COLUMN STANDARDIZATION
   {len(changes)} columns renamed to lowercase with underscores

2. DATE PARSING
   {len(date_cols)} date columns converted to datetime

3. MISSING VALUE TREATMENT
   Dropped {len(cols_to_drop) if cols_to_drop else 0} high-null columns
   Imputed {len(imputation_log) if imputation_log else 0} columns
   Final missing: {final_missing}

4. DUPLICATE REMOVAL
   Removed {n_duplicates:,} duplicate rows

5. OUTLIER TREATMENT
   Capped outliers in {len(capping_log) if capping_log else 0} columns

6. TARGET VARIABLE
   {target_col}: Binary (0=On-time, 1=Late)
   Late rate: {df[target_col].mean()*100:.1f}%

{'='*60}
Output: {output_path}
Next: Run 03_feature_engineering.ipynb
""")


DATA PREPROCESSING COMPLETE

PREPROCESSING PIPELINE SUMMARY

1. COLUMN STANDARDIZATION
   53 columns renamed to lowercase with underscores

2. DATE PARSING
   2 date columns converted to datetime

3. MISSING VALUE TREATMENT
   Dropped 2 high-null columns
   Imputed 2 columns
   Final missing: 0

4. DUPLICATE REMOVAL
   Removed 0 duplicate rows

5. OUTLIER TREATMENT
   Capped outliers in 8 columns

6. TARGET VARIABLE
   late_delivery_risk: Binary (0=On-time, 1=Late)
   Late rate: 54.8%

Output: ../data/interim/cleaned_data_20251205_1120.parquet
Next: Run 03_feature_engineering.ipynb

